# Notebook 46 — Symbolic and Interpretable Governing Equation Extraction

This notebook extracts an explicit, low-complexity equation for the learned residual-flow field

\[
\frac{dr}{dc} = g(r,c)
\]

and compares additive, polynomial, sparse, and interaction-based forms against the observed field
and rollout behavior.

## Notebook chain

- **43**: local nonlinear residual flow exists
- **44**: residual flow is structured, directional, and dissipative
- **45**: the governing field is smooth, low-rank, and nearly additive
- **46**: extract the simplest explicit governing equation consistent with those observations

## Main questions

1. How much of the field is additive: \(g(r,c) \approx f(r) + h(c)\)?
2. What sparse polynomial / interaction terms are really needed?
3. How much interpretability costs in pointwise fit and rollout quality?
4. Does the explicit equation preserve forward dynamics while remaining simple?

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge, LassoCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, SplineTransformer
from sklearn.neural_network import MLPRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split

try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)

In [ ]:
# Data discovery and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.04, 5: 0.02, 7: 0.05}[k]
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                # Smooth, mostly additive field with limited interaction correction
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append(
                                    {
                                        "system": system,
                                        "task": task,
                                        "forcing_mode": forcing_mode,
                                        "k": k,
                                        "flow_mode": flow_mode,
                                        "condition_coord": c,
                                        "residual": r,
                                        "predicted_flow": g + noise,
                                        "next_residual": next_r,
                                        "delta_condition": delta_c,
                                        "sample_id": sample_id,
                                        "path_id": path_id,
                                        "window_id": window_id,
                                        "jacobian_norm": abs(-0.78 + 0.40 * r + nl_gain * 0.10 * c - 0.075 * r**2),
                                        "integration_error": abs(noise),
                                    }
                                )
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

In [ ]:
# Loading + schema alignment

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    # fill required columns if missing
    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        dc = np.gradient(df["condition_coord"].to_numpy())
        df["delta_condition"] = dc

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for k, v in defaults.items():
        if k not in df.columns:
            df[k] = v

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")

    return df

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

## Focus slice

As in Notebook 45, we start with the main slice:

- `system = entropy`
- `task = zeta_vs_gue`
- `forcing_mode = condition_gap`
- `k = 5`

We pick the best available flow mode inside that slice, preferring `nonlinear` if present.

In [ ]:
# Focus selection

focus_mask = (
    (df["system"].astype(str) == "entropy")
    & (df["task"].astype(str) == "zeta_vs_gue")
    & (df["forcing_mode"].astype(str) == "condition_gap")
    & (df["k"].astype(float) == 5)
)

focus_candidates = df[focus_mask].copy()
if focus_candidates.empty:
    # fallback to biggest slice
    group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
    top_group = (
        df.groupby(group_cols)
        .size()
        .reset_index(name="n")
        .sort_values("n", ascending=False)
        .iloc[0]
    )
    focus_mask = np.ones(len(df), dtype=bool)
    for c in group_cols:
        focus_mask &= (df[c].astype(str) == str(top_group[c]))
    focus_candidates = df[focus_mask].copy()

flow_counts = focus_candidates.groupby("flow_mode").size().sort_values(ascending=False)
best_flow_mode = "nonlinear" if "nonlinear" in flow_counts.index else flow_counts.index[0]
focus_df = focus_candidates[focus_candidates["flow_mode"].astype(str) == str(best_flow_mode)].copy()

focus_meta = {
    "system": str(focus_df["system"].iloc[0]),
    "task": str(focus_df["task"].iloc[0]),
    "forcing_mode": str(focus_df["forcing_mode"].iloc[0]),
    "k": focus_df["k"].iloc[0],
    "flow_mode": str(focus_df["flow_mode"].iloc[0]),
}
print("FOCUS:", focus_meta)
print("Focus shape:", focus_df.shape)
display(focus_df.head())

## Gridding and additive decomposition

We grid the observed field over `(condition_coord, residual)` and estimate:

\[
g(r,c) \approx f(r) + h(c) + \epsilon(r,c)
\]

The additive component provides the simplest interpretable structure. The residual interaction field
tells us how much coupling remains after separability.

In [ ]:
# Gridding helpers

def make_gridded_field(data, c_bins=24, r_bins=24):
    sub = data[["condition_coord", "residual", "predicted_flow"]].dropna().copy()
    sub["c_bin"] = pd.cut(sub["condition_coord"], bins=c_bins, labels=False, include_lowest=True)
    sub["r_bin"] = pd.cut(sub["residual"], bins=r_bins, labels=False, include_lowest=True)

    grouped = (
        sub.groupby(["c_bin", "r_bin"])
        .agg(
            mean_flow=("predicted_flow", "mean"),
            count=("predicted_flow", "size"),
            mean_c=("condition_coord", "mean"),
            mean_r=("residual", "mean"),
        )
        .reset_index()
    )
    grid = grouped.pivot(index="r_bin", columns="c_bin", values="mean_flow").sort_index(ascending=True)
    c_centers = grouped.groupby("c_bin")["mean_c"].mean().sort_index()
    r_centers = grouped.groupby("r_bin")["mean_r"].mean().sort_index()
    return grouped, grid, c_centers, r_centers

grouped, grid, c_centers, r_centers = make_gridded_field(focus_df, c_bins=24, r_bins=24)
display(grouped.head())
print("Grid shape:", grid.shape)

In [ ]:
# Additive decomposition on gridded field

G = grid.copy()
overall_mean = np.nanmean(G.values)
col_effect = np.nanmean(G.values - overall_mean, axis=0)
row_effect = np.nanmean(G.values - overall_mean, axis=1)

G_add = overall_mean + row_effect[:, None] + col_effect[None, :]
interaction = G.values - G_add

mask = np.isfinite(G.values)
additive_r2 = 1 - np.nansum((G.values - G_add)**2) / np.nansum((G.values - np.nanmean(G.values))**2)

print(f"Additive approximation R^2 = {additive_r2:.3f}")

In [ ]:
# Plot: empirical field, additive approximation, interaction residual

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

im0 = axes[0].imshow(G.values, aspect="auto", origin="lower", cmap="viridis")
axes[0].set_title("Observed gridded flow")
axes[0].set_xlabel("condition bin")
axes[0].set_ylabel("residual bin")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(G_add, aspect="auto", origin="lower", cmap="viridis")
axes[1].set_title(f"Additive approximation\nR²={additive_r2:.3f}")
axes[1].set_xlabel("condition bin")
axes[1].set_ylabel("residual bin")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(interaction, aspect="auto", origin="lower", cmap="coolwarm")
axes[2].set_title("Interaction residual")
axes[2].set_xlabel("condition bin")
axes[2].set_ylabel("residual bin")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
# Plot additive components f(r), h(c)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(r_centers.values, overall_mean + row_effect, marker="o")
axes[0].set_title("Estimated f(r) + const")
axes[0].set_xlabel("residual r")
axes[0].set_ylabel("flow contribution")

axes[1].plot(c_centers.values, col_effect, marker="o")
axes[1].set_title("Estimated h(c)")
axes[1].set_xlabel("condition coordinate c")
axes[1].set_ylabel("flow contribution")

plt.tight_layout()
plt.show()

## Feature library and interpretable model ladder

We compare a small ladder:

1. additive linear  
2. additive quadratic  
3. additive cubic  
4. interaction polynomial  
5. sparse polynomial library (Lasso)  
6. spline-additive model  

The aim is not pure best fit at any cost, but **best interpretable fit per unit complexity**.

In [ ]:
# Train/test split

model_df = focus_df[["condition_coord", "residual", "predicted_flow"]].dropna().copy()
X = model_df[["residual", "condition_coord"]].to_numpy()
y = model_df["predicted_flow"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.28, random_state=42
)

def rmse(a, b):
    return math.sqrt(mean_squared_error(a, b))

In [ ]:
# Model definitions

models = {
    "additive_linear": Pipeline([
        ("poly", PolynomialFeatures(degree=1, include_bias=True)),
        ("lin", LinearRegression()),
    ]),
    "additive_quadratic_manual": None,   # custom below
    "additive_cubic_manual": None,       # custom below
    "interaction_poly_deg2": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=True)),
        ("lin", LinearRegression()),
    ]),
    "interaction_poly_deg3_ridge": Pipeline([
        ("poly", PolynomialFeatures(degree=3, include_bias=True)),
        ("ridge", Ridge(alpha=1e-3)),
    ]),
    "sparse_poly_lasso": Pipeline([
        ("poly", PolynomialFeatures(degree=3, include_bias=True)),
        ("scale", StandardScaler(with_mean=False)),
        ("lasso", LassoCV(cv=5, random_state=42, max_iter=10000)),
    ]),
    "spline_additive": None,  # custom below
}

In [ ]:
# Custom additive polynomial models

from dataclasses import dataclass

@dataclass
class AdditivePolynomialModel:
    degree: int
    coef_: np.ndarray = None

    def _design(self, X):
        r = X[:, 0]
        c = X[:, 1]
        feats = [np.ones_like(r)]
        for d in range(1, self.degree + 1):
            feats.append(r**d)
        for d in range(1, self.degree + 1):
            feats.append(c**d)
        return np.column_stack(feats)

    def fit(self, X, y):
        Phi = self._design(X)
        beta, *_ = np.linalg.lstsq(Phi, y, rcond=None)
        self.coef_ = beta
        return self

    def predict(self, X):
        return self._design(X) @ self.coef_

    def complexity(self):
        return int(np.sum(np.abs(self.coef_) > 1e-10))

class SplineAdditiveModel:
    def __init__(self, n_knots=5, degree=3):
        self.r_spline = SplineTransformer(n_knots=n_knots, degree=degree, include_bias=False)
        self.c_spline = SplineTransformer(n_knots=n_knots, degree=degree, include_bias=False)
        self.lin = LinearRegression()

    def fit(self, X, y):
        r = X[:, [0]]
        c = X[:, [1]]
        R = self.r_spline.fit_transform(r)
        C = self.c_spline.fit_transform(c)
        Phi = np.hstack([np.ones((len(X), 1)), R, C])
        self.lin.fit(Phi, y)
        self.n_features_ = Phi.shape[1]
        return self

    def predict(self, X):
        r = X[:, [0]]
        c = X[:, [1]]
        R = self.r_spline.transform(r)
        C = self.c_spline.transform(c)
        Phi = np.hstack([np.ones((len(X), 1)), R, C])
        return self.lin.predict(Phi)

    def complexity(self):
        return getattr(self, "n_features_", 0)

models["additive_quadratic_manual"] = AdditivePolynomialModel(degree=2)
models["additive_cubic_manual"] = AdditivePolynomialModel(degree=3)
models["spline_additive"] = SplineAdditiveModel(n_knots=5, degree=3)

In [ ]:
# Fit/evaluate models and extract complexity

def model_complexity(model, name):
    if hasattr(model, "complexity"):
        return model.complexity()
    if name.startswith("interaction_poly"):
        poly = model.named_steps["poly"]
        return len(poly.get_feature_names_out(["r", "c"]))
    if name == "additive_linear":
        return 3
    if name == "sparse_poly_lasso":
        lasso = model.named_steps["lasso"]
        coefs = np.asarray(lasso.coef_, dtype=float)
        max_abs = max(float(np.max(np.abs(coefs))), 1e-12)
        return int(np.sum(np.abs(coefs) > 0.01 * max_abs))
    try:
        return len(model.named_steps["poly"].get_feature_names_out(["r", "c"]))
    except Exception:
        return np.nan

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    res = {
        "model": name,
        "train_r2": r2_score(y_train, pred_train),
        "test_r2": r2_score(y_test, pred_test),
        "train_rmse": rmse(y_train, pred_train),
        "test_rmse": rmse(y_test, pred_test),
        "complexity": model_complexity(model, name),
    }
    results.append(res)
    fitted_models[name] = model

results_df = pd.DataFrame(results).sort_values(["test_r2", "test_rmse"], ascending=[False, True]).reset_index(drop=True)
display(results_df)

In [ ]:
# Pointwise prediction plot

plt.figure(figsize=(7, 5))
for name in results_df["model"].tolist():
    model = fitted_models[name]
    pred = model.predict(X_test)
    plt.scatter(y_test, pred, s=10, alpha=0.55, label=name)

mn = min(y_test.min(), *(fitted_models[n].predict(X_test).min() for n in fitted_models))
mx = max(y_test.max(), *(fitted_models[n].predict(X_test).max() for n in fitted_models))
plt.plot([mn, mx], [mn, mx], linestyle="--")
plt.xlabel("actual flow")
plt.ylabel("predicted flow")
plt.title("Pointwise governing-flow prediction")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Sparse polynomial equation extraction

We now inspect the sparse polynomial model explicitly. This provides a concrete candidate equation
with coefficients attached to named terms.

In [ ]:
# Extract sparse polynomial equation
# Use a relative threshold so active-term selection is stable across scales / k values.

sparse_model = fitted_models["sparse_poly_lasso"]
poly = sparse_model.named_steps["poly"]
lasso = sparse_model.named_steps["lasso"]
feature_names = poly.get_feature_names_out(["r", "c"])
coefs = lasso.coef_
intercept = float(lasso.intercept_)

coef_df = pd.DataFrame({
    "term": feature_names,
    "coefficient": coefs,
    "abs_coefficient": np.abs(coefs),
}).sort_values("abs_coefficient", ascending=False)

max_abs = max(float(coef_df["abs_coefficient"].max()), 1e-12)
active_terms = coef_df[coef_df["abs_coefficient"] > 0.01 * max_abs].copy()
display(active_terms)

def format_equation(intercept, active_df, max_terms=12):
    pieces = [f"{intercept:.4f}"]
    use_df = active_df.head(max_terms)
    for _, row in use_df.iterrows():
        coef = row["coefficient"]
        term = row["term"].replace(" ", "")
        sign = "+" if coef >= 0 else "-"
        pieces.append(f" {sign} {abs(coef):.4f}·{term}")
    return "g(r,c) ≈ " + "".join(pieces)

print(format_equation(intercept, active_terms, max_terms=12))

In [ ]:
# Coefficient importance plot

top_terms = active_terms.head(12).iloc[::-1]
plt.figure(figsize=(8, 5))
plt.barh(top_terms["term"], top_terms["coefficient"])
plt.title("Sparse polynomial term importance")
plt.xlabel("coefficient")
plt.tight_layout()
plt.show()

## Complexity frontier

A useful model is not just accurate — it is accurate **at low complexity**.
We build a simple Pareto-style view: error vs active terms.

In [ ]:
# Complexity frontier plot

plt.figure(figsize=(7, 5))
for _, row in results_df.iterrows():
    plt.scatter(row["complexity"], row["test_rmse"], s=80)
    plt.text(row["complexity"] + 0.1, row["test_rmse"], row["model"], fontsize=8)

plt.xlabel("model complexity")
plt.ylabel("test RMSE")
plt.title("Interpretability / accuracy frontier")
plt.tight_layout()
plt.show()

## Rollout from explicit models

Pointwise fit is not enough. We integrate each candidate equation forward and backward over a canonical
condition grid and compare with the empirical baseline trajectory family.

In [ ]:
# Safe rollout helpers

c_min, c_max = focus_df["condition_coord"].min(), focus_df["condition_coord"].max()
r_min, r_max = focus_df["residual"].min(), focus_df["residual"].max()
flow_q = np.quantile(np.abs(focus_df["predicted_flow"]), 0.995)
FLOW_CAP = float(max(1.0, 2.5 * flow_q))

def safe_predict(model, r, c):
    r_q = float(np.clip(r, r_min, r_max))
    c_q = float(np.clip(c, c_min, c_max))
    x = np.array([[r_q, c_q]], dtype=float)
    val = float(model.predict(x)[0])
    if not np.isfinite(val):
        val = 0.0
    val = float(np.clip(val, -FLOW_CAP, FLOW_CAP))
    return val

def integrate_model(model, r0, c_grid, direction="forward"):
    c_use = np.array(c_grid, dtype=float)
    if direction == "backward":
        c_use = c_use[::-1]
    traj = [float(np.clip(r0, r_min, r_max))]
    r = traj[0]
    for i in range(len(c_use) - 1):
        c = c_use[i]
        dc = float(c_use[i + 1] - c_use[i])
        g = safe_predict(model, r, c)
        r = float(r + g * dc)
        if not np.isfinite(r):
            r = traj[-1]
        r = float(np.clip(r, r_min - 0.5, r_max + 0.5))
        traj.append(r)
    traj = np.array(traj)
    if direction == "backward":
        traj = traj[::-1]
    return traj

# empirical baseline via interpolation from observed starts/ends
canonical_c = np.linspace(c_min, c_max, 60)
r0_values = np.linspace(np.quantile(focus_df["residual"], 0.05), np.quantile(focus_df["residual"], 0.95), 24)

In [ ]:
# Compare forward/backward terminal states

rollout_rows = []
for name in results_df["model"].tolist():
    model = fitted_models[name]
    forward_terms, backward_terms = [], []
    for r0 in r0_values:
        traj_f = integrate_model(model, r0, canonical_c, direction="forward")
        traj_b = integrate_model(model, r0, canonical_c, direction="backward")
        forward_terms.append(traj_f[-1])
        backward_terms.append(traj_b[0])
    rollout_rows.append({
        "model": name,
        "forward_terminal_mean": float(np.mean(forward_terms)),
        "backward_terminal_mean": float(np.mean(backward_terms)),
        "fb_gap_mean": float(np.mean(np.abs(np.array(forward_terms) - np.array(backward_terms)))),
    })

rollout_summary = pd.DataFrame(rollout_rows).sort_values("fb_gap_mean")
display(rollout_summary)

In [ ]:
# Plot forward/backward terminal states for all models

fig, axes = plt.subplots(2, math.ceil(len(results_df) / 2), figsize=(14, 7), sharex=True, sharey=True)
axes = np.array(axes).reshape(-1)

for ax, name in zip(axes, results_df["model"].tolist()):
    model = fitted_models[name]
    forward_terms, backward_terms = [], []
    for r0 in r0_values:
        traj_f = integrate_model(model, r0, canonical_c, direction="forward")
        traj_b = integrate_model(model, r0, canonical_c, direction="backward")
        forward_terms.append(traj_f[-1])
        backward_terms.append(traj_b[0])
    ax.plot(r0_values, forward_terms, label="forward terminal")
    ax.plot(r0_values, backward_terms, label="backward terminal")
    ax.set_title(name)
    ax.set_xlabel("initial residual r0")
    ax.set_ylabel("terminal residual")
    ax.legend(fontsize=7)

for ax in axes[len(results_df):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## Select interpretable winner

We combine:
- pointwise test performance
- model complexity
- forward/backward terminal asymmetry

and identify a **minimal interpretable winner**.

In [ ]:
# Select winner using simple score

score_df = results_df.merge(rollout_summary, on="model", how="left").copy()
score_df["complexity_penalty"] = score_df["complexity"] / score_df["complexity"].max()
score_df["rmse_norm"] = score_df["test_rmse"] / score_df["test_rmse"].max()
score_df["fb_gap_norm"] = score_df["fb_gap_mean"] / score_df["fb_gap_mean"].max()

score_df["interpretable_score"] = (
    0.50 * score_df["rmse_norm"]
    + 0.20 * score_df["fb_gap_norm"]
    + 0.30 * score_df["complexity_penalty"]
)

score_df = score_df.sort_values("interpretable_score", ascending=True).reset_index(drop=True)
display(score_df)

winner_name = score_df.iloc[0]["model"]
winner_model = fitted_models[winner_name]
print("Minimal interpretable winner:", winner_name)

In [ ]:
# Winner governing field on a dense grid

c_dense = np.linspace(c_min, c_max, 140)
r_dense = np.linspace(r_min, r_max, 140)
CC, RR = np.meshgrid(c_dense, r_dense)
field = np.zeros_like(CC)
for i in range(CC.shape[0]):
    for j in range(CC.shape[1]):
        field[i, j] = safe_predict(winner_model, RR[i, j], CC[i, j])

# approximate nullcline from sign changes along r for each c
null_c, null_r = [], []
for j in range(field.shape[1]):
    col = field[:, j]
    for i in range(len(r_dense) - 1):
        if np.isfinite(col[i]) and np.isfinite(col[i+1]) and np.sign(col[i]) != np.sign(col[i+1]):
            r_mid = 0.5 * (r_dense[i] + r_dense[i+1])
            null_c.append(c_dense[j])
            null_r.append(r_mid)

plt.figure(figsize=(8, 6))
im = plt.imshow(
    field,
    aspect="auto",
    origin="lower",
    extent=[c_dense.min(), c_dense.max(), r_dense.min(), r_dense.max()],
    cmap="viridis",
)
plt.colorbar(im, label="predicted g(r,c)")
if len(null_c) > 0:
    plt.scatter(null_c, null_r, s=8, c="purple", label="symbolic nullcline")
plt.xlabel("condition coordinate c")
plt.ylabel("residual r")
plt.title(f"Governing field — {winner_name}")
plt.legend()
plt.tight_layout()
plt.show()

## Symbolic stability map

For an explicit equation, local stability is controlled by:

\[
\frac{\partial g}{\partial r}
\]

We estimate this numerically for the extracted winner and visualize the resulting stability pattern.

In [ ]:
# Numerical derivative wrt r for winner

dr = 1e-3
dgr = np.zeros_like(field)
for i in range(CC.shape[0]):
    for j in range(CC.shape[1]):
        r = RR[i, j]
        c = CC[i, j]
        gp = safe_predict(winner_model, r + dr, c)
        gm = safe_predict(winner_model, r - dr, c)
        dgr[i, j] = (gp - gm) / (2 * dr)

plt.figure(figsize=(8, 6))
im = plt.imshow(
    dgr,
    aspect="auto",
    origin="lower",
    extent=[c_dense.min(), c_dense.max(), r_dense.min(), r_dense.max()],
    cmap="coolwarm",
)
plt.colorbar(im, label="∂g/∂r")
plt.contour(CC, RR, dgr, levels=[0], colors="k", linewidths=1)
plt.xlabel("condition coordinate c")
plt.ylabel("residual r")
plt.title(f"Symbolic stability map — {winner_name}")
plt.tight_layout()
plt.show()

## Trajectory family under the interpretable winner

In [ ]:
# Winner trajectory family

plt.figure(figsize=(8, 5))
for r0 in r0_values:
    traj = integrate_model(winner_model, r0, canonical_c, direction="forward")
    plt.plot(canonical_c, traj, alpha=0.8)
plt.xlabel("condition coordinate c")
plt.ylabel("residual")
plt.title(f"Integrated trajectories — interpretable winner: {winner_name}")
plt.tight_layout()
plt.show()

## Cross-k structural transport

We now test whether the **same equation family** survives across `k = 3, 5, 7` for the same
system/task/forcing/flow slice. We do not require identical coefficients, only comparable active terms.

In [ ]:
# Cross-k sparse fits
# Relative thresholding prevents cross-k sparsity maps from going empty when overall coefficient scales shift.

base_mask = (
    (df["system"].astype(str) == focus_meta["system"])
    & (df["task"].astype(str) == focus_meta["task"])
    & (df["forcing_mode"].astype(str) == focus_meta["forcing_mode"])
    & (df["flow_mode"].astype(str) == focus_meta["flow_mode"])
)

k_term_rows = []
for kval in sorted(df.loc[base_mask, "k"].astype(float).unique()):
    sub = df[base_mask & (df["k"].astype(float) == float(kval))][["condition_coord", "residual", "predicted_flow"]].dropna().copy()
    if len(sub) < 30:
        continue
    Xk = sub[["residual", "condition_coord"]].to_numpy()
    yk = sub["predicted_flow"].to_numpy()

    model = Pipeline([
        ("poly", PolynomialFeatures(degree=3, include_bias=True)),
        ("scale", StandardScaler(with_mean=False)),
        ("lasso", LassoCV(cv=5, random_state=42, max_iter=10000)),
    ])
    model.fit(Xk, yk)
    feats = model.named_steps["poly"].get_feature_names_out(["r", "c"])
    coefs = model.named_steps["lasso"].coef_
    max_abs = max(float(np.max(np.abs(coefs))), 1e-12)
    active_mask = np.abs(coefs) > 0.01 * max_abs
    for term, coef, active in zip(feats, coefs, active_mask):
        k_term_rows.append({"k": kval, "term": term, "coef": coef, "active": bool(active)})

k_term_df = pd.DataFrame(k_term_rows)
active_heat = k_term_df.pivot_table(index="term", columns="k", values="active", aggfunc="max").fillna(False).astype(int)
coef_heat = k_term_df.pivot_table(index="term", columns="k", values="coef", aggfunc="mean").fillna(0.0)

display(active_heat.head(20))

In [ ]:
# Plot: active-term heatmap across k

terms_to_show = active_heat.sum(axis=1).sort_values(ascending=False).head(15).index
plt.figure(figsize=(6, 7))
plt.imshow(active_heat.loc[terms_to_show].values, aspect="auto", cmap="Greys")
plt.yticks(range(len(terms_to_show)), terms_to_show)
plt.xticks(range(len(active_heat.columns)), active_heat.columns)
plt.xlabel("k")
plt.ylabel("term")
plt.title("Sparse term persistence across k")
plt.tight_layout()
plt.show()

## Final summary table

This table combines:
- pointwise fit
- complexity
- rollout asymmetry
- model-selection score

In [ ]:
final_summary = score_df.copy()

def verdict(row):
    if row["model"] == winner_name:
        return "minimal interpretable winner"
    if "additive" in row["model"] and row["test_r2"] > 0.8:
        return "best additive model"
    if "sparse" in row["model"] and row["complexity"] < 8:
        return "interaction-required"
    if row["complexity"] > results_df["complexity"].median() and row["test_r2"] > results_df["test_r2"].max() - 0.01:
        return "overfit but accurate"
    return "too simple" if row["test_r2"] < 0.8 else "competitive"

final_summary["verdict"] = final_summary.apply(verdict, axis=1)
display(final_summary[[
    "model", "complexity", "test_r2", "test_rmse", "fb_gap_mean", "interpretable_score", "verdict"
]])

## Working conclusion

Notebook 46 extracts the governing field into explicit candidate equations and tests whether a simple
interpretable model can preserve the flow learned in Notebook 45.

Typical success pattern:
- high additive fraction
- a small number of retained polynomial / interaction terms
- strong forward rollout
- directional asymmetry retained under backward integration
- persistence of the same sparse term family across nearby `k`

If that pattern holds on the real Notebook 43/45 exports, the next natural step is:

**Notebook 47 — cross-regime coefficient transport and universality**